# CrewAI：用“角色 + 任务”组织 Agent 团队

CrewAI 的思考单位是团队：

- `Agent`：角色、目标、背景故事、可用工具；
- `Task`：具体任务、期望输出；
- `Crew`：把多个 Agent 与 Task 组合起来；
- `Flow`：用事件/状态组织更生产化的流程。

适合：研究、分析、审核、写作、报告生成等多角色协作。

官方文档：https://docs.crewai.com/

项目对应实现：`support_project/adapters/crewai_adapter.py`


In [ ]:
from langchain_cases.common import setup_utf8
from langchain_cases.config import get_settings

setup_utf8()

s = get_settings()
print("模型:", s.chat_model)


## 1. 最小 Crew：一个 Agent + 一个 Task

先看最小结构。`@tool` 的 docstring 和行为会进入模型可见的工具说明。

In [ ]:
from crewai import Agent, Crew, LLM, Process, Task
from crewai.tools import tool

llm = LLM(
    model=f"openai/{s.chat_model}",
    base_url=s.base_url,
    api_key=s.api_key,
    temperature=0,
    max_completion_tokens=500,
)

@tool("order_status")
def order_status(order_id: str) -> str:
    """查询订单状态。"""
    return "已发货" if order_id.upper() == "A1001" else "未知订单"

support_agent = Agent(
    role="订单客服",
    goal="查询订单并简短回答",
    backstory="你只依据工具结果回答。",
    tools=[order_status],
    llm=llm,
    verbose=False,
)

task = Task(
    description="查询订单 A1001 的状态，并用一句话回答。",
    expected_output="一句话订单状态。",
    agent=support_agent,
)
crew = Crew(agents=[support_agent], tasks=[task], process=Process.sequential, verbose=False)
print((await crew.kickoff_async()).raw)


## 2. CrewAI Flow：状态 + 事件

`Flow` 更接近生产编排：`@start` 开始，`@listen` 监听上一步，状态由 `self.state` 管理。
下面是不依赖 LLM 的确定性例子，便于直接运行。

In [ ]:
from pydantic import BaseModel
from crewai.flow.flow import Flow, start, listen

class RefundFlowState(BaseModel):
    order_id: str = ""
    approved: bool = False
    message: str = ""

class RefundFlow(Flow[RefundFlowState]):
    @start()
    def read_order(self):
        self.state.order_id = "A1001"
        return self.state.order_id

    @listen(read_order)
    def decide(self, order_id: str):
        self.state.approved = True
        self.state.message = f"{order_id} 符合质量问题退款规则，需主管复核"

flow = RefundFlow()
await flow.kickoff_async()
print(flow.state)


## 2.1 Flow Router：按条件走不同分支

`@router` 适合做业务分流，例如：

- 金额 ≥ 500 → 主管审批分支；
- 金额 < 500 → 自动通过分支。

下面的例子不调用模型，专门展示 Flow 的状态和路由。

In [ ]:
from typing import Literal
from pydantic import BaseModel
from crewai.flow.flow import Flow, start, listen, router

class RefundRouteState(BaseModel):
    amount: float = 700.0
    route: str = ""

class RefundRouteFlow(Flow[RefundRouteState]):
    @start()
    def inspect_amount(self):
        return "high" if self.state.amount >= 500 else "low"

    @router(inspect_amount)
    def choose_route(self, level: str) -> Literal["auto_approve", "manual_review"]:
        return "manual_review" if level == "high" else "auto_approve"

    @listen("auto_approve")
    def handle_auto_approve(self):
        self.state.route = "自动通过"

    @listen("manual_review")
    def handle_manual_review(self):
        self.state.route = "转主管审批"

route_flow = RefundRouteFlow()
await route_flow.kickoff_async()
print(route_flow.state)


## 3. 售后项目：三角色协作

项目里的 Crew 分工为：

1. 订单与退款分析员：查订单、算退款；
2. 售后政策审核员：检索政策、审核合规性；
3. 客服回复专员：根据前两步写最终回复。

In [ ]:
from support_project.adapters.crewai_adapter import run

result = await run("订单 A1001 的降噪耳机坏了，想退款，是否符合政策？")
print("耗时：", round(result.elapsed_ms / 1000, 2), "s")
print("策略：", result.strategy)
print("\n", result.answer)


## 何时选 CrewAI

- 任务能自然拆成角色，例如研究、审核、写作、校对；
- 你更关心每个角色的目标和产出，而不是每个节点的精确控制；
- 需要较快搭出可读性强的多 Agent 团队；
- 需要 Flow 时，也可以用 `@start / @listen / @router` 做状态流程。

不适合：一个 Agent 就能完成、且对 Token 成本和确定性非常敏感的场景。

项目入口：

```powershell
uv run python -m support_project.runner --framework crewai --full
```
